# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 5_vector_embeddings.ipynb

PURPOSE:
Build semantic embeddings for biomedical papers
to enable fast vector-based retrieval (FAISS).

This is the semantic retrieval layer of ARIA-Lite.

INPUT:
- clean_papers.json (from Week 2)

OUTPUT:
- paper_embeddings.npy
- FAISS index
- pmid mapping

In [1]:
# ============================================================
# SECTION 1 — Install Libraries
# ============================================================
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# SECTION 2 — Imports
# ============================================================

from google.colab import drive
import os
import json
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
import faiss

In [3]:
# ============================================================
# SECTION 3 — Project Paths and data loading
# ============================================================

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite"

PAPER_FILE_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_papers.json")


with open(PAPER_FILE_PATH, "r") as f:
    papers = json.load(f)

print("Papers loaded:", len(papers))

Mounted at /content/drive
Papers loaded: 475


In [4]:
# ============================================================
# SECTION 4 — Extract Text
# ============================================================

pmids = []
texts = []

for p in papers:
    pmids.append(p["pmid"])
    texts.append(p["text"])

print("Example text:\n", texts[0][:300])

Example text:
 Title: Leveraging immune and clinicopathological profiles with machine learning to predict axillary lymph node metastasis in breast cancer patients.. Abstract: BACKGROUND: Breast cancer is the leading cause of cancer-related death in women, with mortality increasing when tumor cells spread to nearby


In [5]:
# ============================================================
# SECTION 5 — Load Embedding Model
# ============================================================

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# ============================================================
# SECTION 6 — Generate Embeddings
# ============================================================

embeddings = []

for text in tqdm(texts):
    emb = model.encode(text)
    embeddings.append(emb)

embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

100%|██████████| 475/475 [01:14<00:00,  6.40it/s]

Embedding shape: (475, 384)


In [7]:
# ============================================================
# SECTION 7 — Normalize and Build FAISS Index
# ============================================================

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index built:", index.ntotal)

FAISS index built: 475


In [8]:
# ============================================================
# SECTION 8 — Query Function
# ============================================================

def search(query, top_k=5):

    query_vec = model.encode([query])
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, top_k)

    results = []

    for i, idx in enumerate(indices[0]):
        results.append({
            "pmid": pmids[idx],
            "text": texts[idx],
            "score": float(scores[0][i])
        })

    return results

In [9]:
results = search("breast cancer immunotherapy", top_k=5)

for r in results:
    print("\nPMID:", r["pmid"])
    print("Score:", r["score"])
    print(r["text"][:300])


PMID: 42082270
Score: 0.580161988735199
Title: Safety and immunogenicity of a tri-antigen vaccine targeting IGFBP-2, HER2, and IGF-IR in participants with non-metastatic breast cancer.. Abstract: BACKGROUND: Ductal carcinoma in situ (DCIS) is a preinvasive form of breast cancer. Current treatment consists of surgery, radiation, and often 

PMID: 42099744
Score: 0.5660370588302612
Title: Immunotherapy innovations in triple-negative breast cancer: targeting checkpoints, combinations, and biomarkers.. Abstract: UNLABELLED: Triple-negative breast cancer (TNBC), an aggressive subtype lacking estrogen receptor (ER), progesterone receptor (PR), and HER2 expression, accounts for 10-

PMID: 41988184
Score: 0.5135576725006104
Title: Iodine-125 brachytherapy triggers immunogenic cell death and potentiates anti-PD-L1 immunotherapy in bone metastatic triple-negative breast cancer.. Abstract: INTRODUCTION: Triple-negative breast cancer (TNBC) with bone metastasis is challenging to treat due to an i

In [10]:
import os
import pickle
import faiss

BASE_DIR = os.path.join(PROJECT_ROOT, "data", "indices")
os.makedirs(BASE_DIR, exist_ok=True)

# -----------------------------
# Save FAISS index
# -----------------------------
faiss.write_index(index, os.path.join(BASE_DIR, "faiss_index.bin"))

# -----------------------------
# Save metadata
# -----------------------------
with open(os.path.join(BASE_DIR, "pmids.pkl"), "wb") as f:
    pickle.dump(pmids, f)

with open(os.path.join(BASE_DIR, "texts.pkl"), "wb") as f:
    pickle.dump(texts, f)

print("Vector index saved in data/indices/")

Vector index saved in data/indices/
